<a href="https://colab.research.google.com/github/Alfonsotomax/Adfonfin/blob/main/notebook/AVE_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
import torch
import numpy as np
import torchvision.transforms.v2 as T
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os
from google.colab import drive
from torch.utils.data import TensorDataset, DataLoader


# Runtime → Change runtime type → T4 GPU for faster training compared to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

drive.mount('/content/drive')

!cp '/content/drive/MyDrive/dog-project-data/processed/dogs64.npy' '/content/'
df = np.load('/content/dogs64.npy')
csv_data = pd.read_csv('/content/drive/MyDrive/dog-project-data/processed/metadata.csv')





Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Getting the index of the training, val & test data
train = csv_data.index[csv_data["split"] == "train"].tolist()
val   = csv_data.index[csv_data["split"] == "val"].tolist()
test  = csv_data.index[csv_data["split"] == "test"].tolist()

# getting the actual images from the indexs
train_images = df[train]
val_images = df[val]
test_images = df[test]

del df # free up memory

# Converting the 0...255 images to float 32
#train_images = train_images.astype(np.float32)
#val_images = val_images.astype(np.float32)

# mapping 0..255 to [-1,1] for tanh
#train_images = (train_images / 127.5) - 1
#val_images = (val_images / 127.5) - 1


In [ ]:
#print(train_images.shape) #(72473, 64, 64, 3) need (N, 3, 64, 64)
train_images = train_images.transpose(0, 3, 1, 2)
val_images = val_images.transpose(0, 3, 1, 2)
test_images = test_images.transpose(0, 3, 1, 2)
#print(val_images.shape)

train_images = np.ascontiguousarray(train_images)
val_images = np.ascontiguousarray(val_images)
test_images = np.ascontiguousarray(test_images)

# convert to tensor from numpy array
train_images = torch.from_numpy(train_images)
val_images = torch.from_numpy(val_images)
test_images = torch.from_numpy(test_images)


In [ ]:
# Need TensorDataset
train_dataset = TensorDataset(train_images)
val_dataset = TensorDataset(val_images)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, drop_last = True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False, drop_last = False)



#(a, ) = next(iter(train_loader))
#a = a.to(device)
#a = a.float() / 127.5 - 1.0

# uint8 [0, 255] on CPU -> float [-1, 1] on the GPU
def prep_batch(batch):
  return batch.to(device, non_blocking=True).float() / 127.5 - 1.0

#print(a.shape)
#print(a.dtype)
#print(a.device)
#print(a.min())
#print(a.max())

#for (batch, ) in train_loader:
  #batch = batch.to(device)

In [39]:
def show_grid(batch):
  data = batch[:64]
  rescale = (data + 1) * 127.5
  rescale = rescale.clamp(0,255)
  rescale = rescale.byte()
  rescale = rescale.permute(0,2,3,1)
  rescale = rescale.detach().cpu().numpy()
  grid = rescale.reshape(8, 8, 64, 64, 3).transpose(0, 2, 1, 3, 4).reshape(8 * 64, 8 * 64, 3)
  plt.figure(figsize=(10, 10))
  plt.imshow(grid)
  plt.axis('off')
  plt.show()

In [40]:
class Encoder(nn.Module):
  # runs once when you create the Neural Network architecture
  def __init__(self, latent_dim):
    super().__init__()
    self.convs = nn.Sequential(
      # 3x64x64 -> 32x32x32
      nn.Conv2d(3, 32, 4, stride=2, padding=1),
      nn.BatchNorm2d(32),
      nn.LeakyReLU(0.2),

      # 32x32x32 -> 64x16x16
      nn.Conv2d(32, 64, 4, stride=2, padding=1),
      nn.BatchNorm2d(64),
      nn.LeakyReLU(0.2),

      # 64x16x16 -> 128x8x8
      nn.Conv2d(64, 128, 4, stride=2, padding=1),
      nn.BatchNorm2d(128),
      nn.LeakyReLU(0.2),

      # 128x8x8 -> 256x4x4
      nn.Conv2d(128, 256, 4, stride=2, padding=1),
      nn.BatchNorm2d(256),
      nn.LeakyReLU(0.2),
    )
    self.fc_logvar = nn.Linear(in_features=256 * 4 * 4, out_features=latent_dim) # learning for the log variance of the latent space
    self.fc_mu = nn.Linear(in_features=256 * 4 * 4, out_features=latent_dim) # learning for the mean of the latent space

  # runs on every batch
  def forward(self, x):
    x = self.convs(x)
    x = torch.flatten(x, start_dim=1)
    mu = self.fc_mu(x)
    logvar = self.fc_logvar(x)
    return mu, logvar


In [45]:
class Decoder(nn.Module):
  def __init__(self, latent_dim):
    super().__init__()
    # z (128 numbers) -> 4096 numbers, which get reshaped to 256x4x4
    self.fc = nn.Linear(in_features=latent_dim, out_features=256 * 4 * 4)

    self.deconvs = nn.Sequential(
      # 256x4x4 -> 128x8x8
      nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1),
      nn.BatchNorm2d(128),
      nn.ReLU(),

      # 128x8x8 -> 64x16x16
      nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
      nn.BatchNorm2d(64),
      nn.ReLU(),

      # 64x16x16 -> 32x32x32
      nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
      nn.BatchNorm2d(32),
      nn.ReLU(),

      # 32x32x32 -> 3x64x64  (last layer: no BatchNorm, Tanh for [-1, 1])
      nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
      nn.Tanh(),
    )

  def forward(self, z):
    x = self.fc(z)               # (B, 128) -> (B, 4096)
    x = x.view(-1, 256, 4, 4)    # (B, 4096) -> (B, 256, 4, 4)
    x = self.deconvs(x)          # (B, 256, 4, 4) -> (B, 3, 64, 64)
    return x

In [ ]:
class VAE(nn.Module):
  def __init__(self, latent_dim):
    super().__init__()
    self.latent_dim = latent_dim
    self.encoder = Encoder(latent_dim)
    self.decoder = Decoder(latent_dim)

  def forward(self, x):
    mu, logvar = self.encoder(x)
    z = self.reparameterize(mu, logvar)
    recon = self.decoder(z)
    return recon, mu, logvar

  def reparameterize(self, mu, logvar):
    std = torch.exp(0.5 * logvar)  # log(sigma^2) -> sigma
    epsilon = torch.randn_like(std)
    return mu + std * epsilon

  @torch.no_grad()
  def sample(self, n):
    z = torch.randn(n, self.latent_dim, device=next(self.parameters()).device)
    return self.decoder(z)


In [ ]:
def VAE_loss(recon, x, mu, logvar, beta=1.0):
  batch_size = x.size(0)

  # Kl is the divergence between N(mu, var^2) and N(0,1)
  kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp()) / batch_size

  # Mean square error as the loss function
  recon_loss = F.mse_loss(recon, x, reduction="sum") / batch_size

  total = recon_loss + beta * kl
  
  return total, recon_loss, kl

In [ ]:
!pip install -q pytorch-fid

import json
from pytorch_fid.inception import InceptionV3
from pytorch_fid.fid_score import calculate_frechet_distance

torch.manual_seed(42)

inception = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()

@torch.no_grad()
def inception_features(images, batch_size=250):
  """images: uint8 tensor (N, 3, 64, 64). Returns (N, 2048) numpy array."""
  feats = []
  for i in range(0, len(images), batch_size):
    x = images[i:i + batch_size].to(device).float() / 255.0  
    feats.append(inception(x)[0].squeeze(-1).squeeze(-1).cpu().numpy())
  return np.concatenate(feats)

def fid_stats(feats):
  return np.mean(feats, axis=0), np.cov(feats, rowvar=False)

def fid_score(images, ref_stats):
  mu, sigma = fid_stats(inception_features(images))
  return float(calculate_frechet_distance(mu, sigma, *ref_stats))

@torch.no_grad()
def generate(model, n, batch_size=500):
  """n new images from z ~ N(0, I), as uint8 (N, 3, 64, 64) on the CPU."""
  model.eval()
  out = []
  for i in range(0, n, batch_size):
    x = model.sample(min(batch_size, n - i))
    out.append(((x + 1) * 127.5).clamp(0, 255).round().to(torch.uint8).cpu())
  return torch.cat(out)

In [ ]:
# Per-epoch reference: the validation set (computed once)
val_stats = fid_stats(inception_features(val_images))
print("val reference:", len(val_images), "images")

In [ ]:
LATENT_DIM = 128
EPOCHS = 25
KL_WARMUP = 5  # epochs to ramp beta up to 1
FID_SAMPLES_EPOCH = 5000
FID_SAMPLES_FINAL = 10000

PROJECT_DIR = "/content/drive/MyDrive/dog-project-data"
CKPT_PATH = f"{PROJECT_DIR}/checkpoints/vae/last.pt"
RESULTS_DIR = f"{PROJECT_DIR}/results/vae"
SAMPLES_DRIVE_DIR = f"{PROJECT_DIR}/midterm generated samples/vae"   # final 10,000 PNGs 
os.makedirs(os.path.dirname(CKPT_PATH), exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

vae = VAE(LATENT_DIM).to(device)
optimizer = torch.optim.Adam(vae.parameters(), lr=1e-3)

start_epoch = 0
history = []
fixed_z = torch.randn(64, LATENT_DIM, device=device)  # same z every epoch

# resume if a checkpoint exists 
if os.path.exists(CKPT_PATH):
  ckpt = torch.load(CKPT_PATH, map_location=device)
  vae.load_state_dict(ckpt["model"])
  optimizer.load_state_dict(ckpt["optimizer"])
  start_epoch = ckpt["epoch"] + 1
  history = ckpt["history"]
  fixed_z = ckpt["fixed_z"]
  print(f"resumed from epoch {start_epoch}")

def plot_fid(history, path=None):
  epochs = [h["epoch"] for h in history]   
  fids = [h["fid_val"] for h in history]
  plt.figure(figsize=(7, 4))
  plt.plot(epochs, fids, marker="o")
  plt.xlabel("Epoch")
  plt.ylabel(f"FID vs val ({FID_SAMPLES_EPOCH:,} samples)")
  plt.title("VAE: FID vs epoch")
  plt.grid(alpha=0.3)
  if path:
    plt.savefig(path, dpi=150, bbox_inches="tight")

In [ ]:
for epoch in range(start_epoch, EPOCHS):
  beta = min(1.0, (epoch + 1) / KL_WARMUP)  # KL warm-up

  # train 
  vae.train()
  train_recon = train_kl = 0.0
  for (batch,) in train_loader:
    x = prep_batch(batch)

    # random horizontal flip on about half the images
    flip = torch.rand(x.size(0), 1, 1, 1, device=device) < 0.5
    x = torch.where(flip, x.flip(3), x)

    recon, mu, logvar = vae(x)
    loss, recon_loss, kl = VAE_loss(recon, x, mu, logvar, beta)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_recon += recon_loss.item()
    train_kl += kl.item()

  train_recon /= len(train_loader)
  train_kl /= len(train_loader)

  # validate (beta = 1 so epochs are comparable) 
  vae.eval()
  val_recon = val_kl = 0.0
  with torch.no_grad():
    for (batch,) in val_loader:
      x = prep_batch(batch)
      recon, mu, logvar = vae(x)
      _, recon_loss, kl = VAE_loss(recon, x, mu, logvar)
      val_recon += recon_loss.item()
      val_kl += kl.item()
  val_recon /= len(val_loader)
  val_kl /= len(val_loader)

  # FID: 5,000 samples vs the validation set 
  fid_val = fid_score(generate(vae, FID_SAMPLES_EPOCH), val_stats)

  history.append({"epoch": epoch, "beta": beta,
                  "train_recon": train_recon, "train_kl": train_kl,
                  "val_recon": val_recon, "val_kl": val_kl, "fid_val": fid_val})
  print(f"epoch {epoch:02d} | beta {beta:.2f} | train recon {train_recon:.1f} kl {train_kl:.1f} "
        f"| val recon {val_recon:.1f} kl {val_kl:.1f} | FID {fid_val:.2f}")

  # checkpoint + FID graph saved every epoch 
  torch.save({"model": vae.state_dict(), "optimizer": optimizer.state_dict(),
              "epoch": epoch, "history": history, "fixed_z": fixed_z}, CKPT_PATH)
  plot_fid(history, f"{RESULTS_DIR}/fid_vs_epoch.png")
  plt.close()

  # look at samples every 5 epochs 
  if (epoch + 1) % 5 == 0:
    with torch.no_grad():
      show_grid(vae.decoder(fixed_z))

In [ ]:
# FID vs epoch graph 
plot_fid(history)
plt.show()

pd.DataFrame(history).round(2)

In [ ]:
import shutil
from PIL import Image

# reference statistics
trainval_stats = fid_stats(np.concatenate([inception_features(train_images), inception_features(val_images)]))
test_stats = fid_stats(inception_features(test_images))

# 10,000 final samples, scored against both references
torch.manual_seed(0)
final_images = generate(vae, FID_SAMPLES_FINAL)
final_mu, final_sigma = fid_stats(inception_features(final_images))

fid_trainval = float(calculate_frechet_distance(final_mu, final_sigma, *trainval_stats))
fid_test = float(calculate_frechet_distance(final_mu, final_sigma, *test_stats))

# Save 10,000 PNGs to drive 
FINAL_GEN_DIR = "/content/final_gen_vae"
shutil.rmtree(FINAL_GEN_DIR, ignore_errors=True)
os.makedirs(FINAL_GEN_DIR)
for i, img in enumerate(final_images.permute(0, 2, 3, 1).numpy()):   # CHW -> HWC for PIL
  Image.fromarray(img).save(f"{FINAL_GEN_DIR}/{i:05d}.png")
shutil.copytree(FINAL_GEN_DIR, SAMPLES_DRIVE_DIR, dirs_exist_ok=True)

# save weights and results to Drive
torch.save(vae.state_dict(), f"{RESULTS_DIR}/vae_final.pt")
results = {"model": "VAE", "epochs": EPOCHS, "latent_dim": LATENT_DIM, "n_samples": FID_SAMPLES_FINAL,
           "fid_train_val": fid_trainval, "fid_test": fid_test, "history": history}
with open(f"{RESULTS_DIR}/results.json", "w") as f:
  json.dump(results, f, indent=2)

print(f"VAE final FID vs train+val ({len(train_images) + len(val_images):,} imgs): {fid_trainval:.2f}")
print(f"VAE final FID vs test      ({len(test_images):,} imgs): {fid_test:.2f}")
print(f"{len(os.listdir(SAMPLES_DRIVE_DIR)):,} PNGs saved to", SAMPLES_DRIVE_DIR)